# Imports and configuration

In [ ]:
!pip install torch-geometric

In [4]:
# ============================================================================
# GENERACIÓN DE GRAFOS TEMPORALES DINÁMICOS PARA NIDS
# Versión Final con manejo correcto de protocolos y centinelas
# ============================================================================

import pandas as pd
import numpy as np
import pickle
import torch
from torch_geometric.data import Data
from collections import defaultdict
from tqdm import tqdm
import time
import os

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

# Paths (AJUSTAR según tu Google Drive)
CSV_PATH = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/f78acbaa2afe1595_NFV3DATA-A11964_A11964/data/NF-CICIDS2018-v3.csv'
FILTERED_CHUNKS_FILE = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/filtered_chunks.pkl'
OUTPUT_DIR = '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_DYNAMIC/'

# Crear directorio de salida
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Columnas del dataset
TIMESTAMP_COL = 'FLOW_START_MILLISECONDS'
SRC_IP_COL = 'IPV4_SRC_ADDR'
DST_IP_COL = 'IPV4_DST_ADDR'
LABEL_COL = 'Attack'

# Parámetros de ventanas temporales
WINDOW_SIZE = pd.Timedelta(minutes=10)   # Tumbling windows de 10 min
WINDOW_STEP = pd.Timedelta(minutes=10)   # Sin overlap

# Procesamiento
CHUNK_SIZE = 250000

# Números de protocolo IP (IANA standard)
IPPROTO_ICMP = 1      # Internet Control Message Protocol
IPPROTO_TCP = 6       # Transmission Control Protocol
IPPROTO_UDP = 17      # User Datagram Protocol
IPPROTO_ICMPV6 = 58   # ICMP for IPv6

print("="*70)
print("📋 CONFIGURACIÓN")
print("="*70)
print(f"Window size: {WINDOW_SIZE}")
print(f"Window step: {WINDOW_STEP}")
print(f"Output dir: {OUTPUT_DIR}")
print("="*70)

📋 CONFIGURACIÓN
Window size: 0 days 00:10:00
Window step: 0 days 00:10:00
Output dir: /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/graph_data_DYNAMIC/


In [5]:
# Cambiar al directorio
os.chdir('/content/drive/MyDrive/nids-mitre/')

# Verificar que estás ahí
!pwd

# Ver qué archivos .py tenés
!ls

/content/drive/MyDrive/nids-mitre
aws  awscliv2.zip  code  data  data_unsw  output  plots  __pycache__  results


In [11]:
# ============================================================================
# MANTENER SESIÓN ACTIVA (Opcional pero recomendado)
# ============================================================================

from IPython.display import Javascript

display(Javascript('''
    function ClickConnect(){
        console.log("Manteniendo conexión activa...");
        document.querySelector("#top-toolbar > colab-connect-button")?.shadowRoot.querySelector("#connect")?.click();
    }
    setInterval(ClickConnect, 60000);
'''))

print("✅ Auto-click activado para mantener sesión")

<IPython.core.display.Javascript object>

✅ Auto-click activado para mantener sesión


# Read chunks

In [6]:
# ============================================================================
# FUNCIÓN: Leer chunks filtrados
# ============================================================================

def read_filtered_chunks(pickle_file):
    """
    Lee chunks desde pickle con manejo de memoria
    """
    with open(pickle_file, 'rb') as f:
        while True:
            try:
                chunk = pickle.load(f)
                yield chunk
            except EOFError:
                break

# Node features

In [7]:
# ============================================================================
# FUNCIÓN: Calcular NODE FEATURES (enriquecidas)
# ============================================================================

def compute_node_features(ip, window_df, src_col, dst_col):
    """
    Calcula features enriquecidas para un nodo (IP)

    Returns:
        list: 18 features por nodo
    """

    # Flows donde esta IP participa
    as_src = window_df[window_df[src_col] == ip]
    as_dst = window_df[window_df[dst_col] == ip]
    all_flows = window_df[(window_df[src_col] == ip) | (window_df[dst_col] == ip)]

    # 1. Conteos básicos
    total_flows = len(all_flows)
    flows_as_src = len(as_src)
    flows_as_dst = len(as_dst)

    # 2. Grados (deg_in, deg_out)
    deg_in = as_dst[src_col].nunique() if len(as_dst) > 0 else 0
    deg_out = as_src[dst_col].nunique() if len(as_src) > 0 else 0

    # 3. Volumen (bytes)
    bytes_out = as_src['OUT_BYTES'].sum() if len(as_src) > 0 else 0
    bytes_in = as_dst['IN_BYTES'].sum() if len(as_dst) > 0 else 0
    ratio_out_in = bytes_out / (bytes_in + 1e-6)

    # 4. Volumen (packets)
    pkts_out = as_src['OUT_PKTS'].sum() if len(as_src) > 0 else 0
    pkts_in = as_dst['IN_PKTS'].sum() if len(as_dst) > 0 else 0

    # 5. Diversidad
    num_peers = deg_in + deg_out

    if 'L4_SRC_PORT' in as_src.columns:
        ports_src = as_src['L4_SRC_PORT'].nunique() if len(as_src) > 0 else 0
    else:
        ports_src = 0

    if 'L4_DST_PORT' in as_dst.columns:
        ports_dst = as_dst['L4_DST_PORT'].nunique() if len(as_dst) > 0 else 0
    else:
        ports_dst = 0

    # 6. Distribución de protocolos
    if len(all_flows) > 0:
        tcp_ratio = (all_flows['PROTOCOL'] == IPPROTO_TCP).sum() / len(all_flows)
        udp_ratio = (all_flows['PROTOCOL'] == IPPROTO_UDP).sum() / len(all_flows)
        icmp_ratio = ((all_flows['PROTOCOL'] == IPPROTO_ICMP) |
                      (all_flows['PROTOCOL'] == IPPROTO_ICMPV6)).sum() / len(all_flows)
    else:
        tcp_ratio = udp_ratio = icmp_ratio = 0.0

    # 7. Estadísticas temporales
    if len(all_flows) > 0:
        avg_duration = all_flows['FLOW_DURATION_MILLISECONDS'].mean()
        std_duration = all_flows['FLOW_DURATION_MILLISECONDS'].std()
        if pd.isna(std_duration):
            std_duration = 0.0
    else:
        avg_duration = std_duration = 0.0

    # Ensamblar vector de features
    features = [
        total_flows,      # 0
        flows_as_src,     # 1
        flows_as_dst,     # 2
        bytes_out,        # 3
        bytes_in,         # 4
        ratio_out_in,     # 5
        pkts_out,         # 6
        pkts_in,          # 7
        deg_in,           # 8
        deg_out,          # 9
        num_peers,        # 10
        ports_src,        # 11
        ports_dst,        # 12
        tcp_ratio,        # 13
        udp_ratio,        # 14
        icmp_ratio,       # 15
        avg_duration,     # 16
        std_duration,     # 17
    ]

    return features

# Edge features

In [8]:
# ============================================================================
# FUNCIÓN: Calcular EDGE FEATURES (enriquecidas con manejo de centinelas)
# ============================================================================

def compute_edge_features(flows_df):
    """
    Calcula features agregadas para una arista (src→dst)
    Maneja correctamente valores centinela por protocolo

    Returns:
        list: 37 features por edge
    """

    n_flows = len(flows_df)

    # ════════════════════════════════════════════════════════════
    # 1. IDENTIFICAR PROTOCOLOS (basado en PROTOCOL)
    # ════════════════════════════════════════════════════════════

    is_tcp = (flows_df['PROTOCOL'] == IPPROTO_TCP)
    is_udp = (flows_df['PROTOCOL'] == IPPROTO_UDP)
    is_icmp = (flows_df['PROTOCOL'] == IPPROTO_ICMP)
    is_icmpv6 = (flows_df['PROTOCOL'] == IPPROTO_ICMPV6)

    # DNS: basado en QUERY_TYPE (captura tunneling)
    is_dns = (flows_df['DNS_QUERY_TYPE'] != 0) if 'DNS_QUERY_TYPE' in flows_df.columns else pd.Series([False]*len(flows_df))

    # ICMP combinado (IPv4 + IPv6)
    has_icmp = is_icmp | is_icmpv6

    # ════════════════════════════════════════════════════════════
    # 2. AGREGAR CON FILTROS (solo flows donde campo aplica)
    # ════════════════════════════════════════════════════════════

    # ────────────────────────────────────────────────────────────
    # DNS
    # ────────────────────────────────────────────────────────────
    dns_flows = flows_df[is_dns]
    if len(dns_flows) > 0 and 'DNS_TTL_ANSWER' in dns_flows.columns:
        dns_ttl_mean = dns_flows['DNS_TTL_ANSWER'].mean()
        dns_ttl_std = dns_flows['DNS_TTL_ANSWER'].std()
        if pd.isna(dns_ttl_std):
            dns_ttl_std = 0.0

        dns_query_types = dns_flows['DNS_QUERY_TYPE'].value_counts()
        dns_query_type_mode = dns_query_types.index[0] if len(dns_query_types) > 0 else 0

        edge_has_dns = 1.0
        dns_ratio = len(dns_flows) / n_flows
    else:
        dns_ttl_mean = dns_ttl_std = dns_query_type_mode = 0.0
        edge_has_dns = dns_ratio = 0.0

    # ────────────────────────────────────────────────────────────
    # ICMP (SOLO flows con PROTOCOL = 1 o 58)
    # ────────────────────────────────────────────────────────────
    icmp_flows = flows_df[has_icmp]
    if len(icmp_flows) > 0 and 'ICMP_TYPE' in icmp_flows.columns:
        icmp_types_raw = icmp_flows['ICMP_TYPE']
        icmp_types = icmp_types_raw // 256
        icmp_codes = icmp_types_raw % 256

        icmp_type_mode = icmp_types.mode()[0] if len(icmp_types) > 0 else 0
        icmp_code_mode = icmp_codes.mode()[0] if len(icmp_codes) > 0 else 0

        edge_has_icmp = 1.0
        icmp_ratio = len(icmp_flows) / n_flows
        edge_has_icmpv4 = 1.0 if is_icmp.any() else 0.0
        edge_has_icmpv6 = 1.0 if is_icmpv6.any() else 0.0
    else:
        icmp_type_mode = icmp_code_mode = 0.0
        edge_has_icmp = icmp_ratio = 0.0
        edge_has_icmpv4 = edge_has_icmpv6 = 0.0

    # ────────────────────────────────────────────────────────────
    # TCP
    # ────────────────────────────────────────────────────────────
    tcp_flows = flows_df[is_tcp]
    if len(tcp_flows) > 0:
        if 'TCP_FLAGS' in tcp_flows.columns:
            tcp_flags_counts = tcp_flows['TCP_FLAGS'].value_counts()
            tcp_flags_mode = tcp_flags_counts.index[0] if len(tcp_flags_counts) > 0 else 0
        else:
            tcp_flags_mode = 0.0

        if 'TCP_WIN_MAX_IN' in tcp_flows.columns:
            tcp_win_mean = tcp_flows['TCP_WIN_MAX_IN'].mean()
            tcp_win_max = tcp_flows['TCP_WIN_MAX_IN'].max()
        else:
            tcp_win_mean = tcp_win_max = 0.0

        edge_has_tcp = 1.0
        tcp_ratio = len(tcp_flows) / n_flows
    else:
        tcp_flags_mode = tcp_win_mean = tcp_win_max = 0.0
        edge_has_tcp = tcp_ratio = 0.0

    # ────────────────────────────────────────────────────────────
    # UDP
    # ────────────────────────────────────────────────────────────
    edge_has_udp = 1.0 if is_udp.any() else 0.0
    udp_ratio = is_udp.sum() / n_flows

    # ════════════════════════════════════════════════════════════
    # 3. FEATURES BÁSICAS (siempre aplicables)
    # ════════════════════════════════════════════════════════════

    duration_mean = flows_df['FLOW_DURATION_MILLISECONDS'].mean()
    duration_std = flows_df['FLOW_DURATION_MILLISECONDS'].std()
    duration_min = flows_df['FLOW_DURATION_MILLISECONDS'].min()
    duration_max = flows_df['FLOW_DURATION_MILLISECONDS'].max()

    bytes_in_sum = flows_df['IN_BYTES'].sum()
    bytes_out_sum = flows_df['OUT_BYTES'].sum()
    bytes_total = bytes_in_sum + bytes_out_sum
    bytes_ratio = bytes_out_sum / (bytes_in_sum + 1e-6)

    pkts_in_sum = flows_df['IN_PKTS'].sum()
    pkts_out_sum = flows_df['OUT_PKTS'].sum()
    pkts_total = pkts_in_sum + pkts_out_sum

    avg_pkt_size = bytes_total / (pkts_total + 1e-6)

    # ════════════════════════════════════════════════════════════
    # 4. FEATURES ADICIONALES (si existen)
    # ════════════════════════════════════════════════════════════

    # IAT
    if 'SRC_TO_DST_IAT_AVG' in flows_df.columns:
        iat_src_dst_mean = flows_df['SRC_TO_DST_IAT_AVG'].mean()
        iat_src_dst_std = flows_df['SRC_TO_DST_IAT_STDDEV'].mean()
        iat_dst_src_mean = flows_df['DST_TO_SRC_IAT_AVG'].mean()
        iat_dst_src_std = flows_df['DST_TO_SRC_IAT_STDDEV'].mean()
    else:
        iat_src_dst_mean = iat_src_dst_std = 0.0
        iat_dst_src_mean = iat_dst_src_std = 0.0

    # TTL
    if 'MIN_TTL' in flows_df.columns:
        ttl_min = flows_df['MIN_TTL'].min()
        ttl_max = flows_df['MAX_TTL'].max()
        ttl_mean = flows_df['MIN_TTL'].mean()
    else:
        ttl_min = ttl_max = ttl_mean = 0.0

    # Retransmisiones
    if 'RETRANSMITTED_IN_PKTS' in flows_df.columns:
        retrans_in = flows_df['RETRANSMITTED_IN_PKTS'].sum()
        retrans_out = flows_df['RETRANSMITTED_OUT_PKTS'].sum()
    else:
        retrans_in = retrans_out = 0.0

    # ════════════════════════════════════════════════════════════
    # 5. ENSAMBLAR VECTOR (handle NaN/inf)
    # ════════════════════════════════════════════════════════════

    def safe(x):
        if pd.isna(x) or np.isnan(x) or np.isinf(x):
            return 0.0
        return float(x)

    features = [
        # Básicas (0-9)
        safe(bytes_in_sum),         # 0
        safe(bytes_out_sum),        # 1
        safe(bytes_ratio),          # 2
        safe(pkts_in_sum),          # 3
        safe(pkts_out_sum),         # 4
        safe(avg_pkt_size),         # 5
        safe(duration_mean),        # 6
        safe(duration_std),         # 7
        safe(duration_min),         # 8
        safe(duration_max),         # 9

        # Retransmisiones (10-11)
        safe(retrans_in),           # 10
        safe(retrans_out),          # 11

        # IAT (12-15)
        safe(iat_src_dst_mean),     # 12
        safe(iat_src_dst_std),      # 13
        safe(iat_dst_src_mean),     # 14
        safe(iat_dst_src_std),      # 15

        # TTL (16-18)
        safe(ttl_min),              # 16
        safe(ttl_max),              # 17
        safe(ttl_mean),             # 18

        # DNS (19-21)
        safe(dns_ttl_mean),         # 19
        safe(dns_ttl_std),          # 20
        safe(dns_query_type_mode),  # 21

        # ICMP (22-23) - SOLO de flows PROTOCOL=1/58
        safe(icmp_type_mode),       # 22
        safe(icmp_code_mode),       # 23

        # TCP (24-26)
        safe(tcp_flags_mode),       # 24
        safe(tcp_win_mean),         # 25
        safe(tcp_win_max),          # 26

        # FLAGS binarios (27-31)
        edge_has_tcp,               # 27
        edge_has_udp,               # 28
        edge_has_icmpv4,            # 29
        edge_has_icmpv6,            # 30
        edge_has_dns,               # 31

        # Ratios de protocolo (32-35)
        tcp_ratio,                  # 32
        udp_ratio,                  # 33
        icmp_ratio,                 # 34
        dns_ratio,                  # 35

        # Metadata (36)
        n_flows,                    # 36
    ]

    return features  # Total: 37 features

# Dynamic graphs

In [9]:
# ============================================================================
# FUNCIÓN: Crear grafo con NODOS DINÁMICOS + Trazabilidad
# ============================================================================

def create_graph_dynamic(window_df, window_id, window_start, window_end,
                        src_col, dst_col, label_col, global_ip_to_id):
    """
    Crea grafo con SOLO nodos activos (dinámico) + trazabilidad

    Returns:
        Data: PyTorch Geometric Data object o None si ventana vacía
    """

    # IPs activas en ESTA ventana
    active_ips = set(window_df[src_col].unique()) | set(window_df[dst_col].unique())
    active_ips = sorted(list(active_ips))

    if len(active_ips) == 0:
        return None

    # Mapeo LOCAL (para este grafo)
    local_ip_to_idx = {ip: idx for idx, ip in enumerate(active_ips)}

    # Mapeo a global_id (para trazabilidad)
    local_to_global = {idx: global_ip_to_id[ip] for idx, ip in enumerate(active_ips)}

    # ────────────────────────────────────────────────────────────
    # NODE FEATURES
    # ────────────────────────────────────────────────────────────
    node_features = []
    for ip in active_ips:
        features = compute_node_features(ip, window_df, src_col, dst_col)
        node_features.append(features)

    # ────────────────────────────────────────────────────────────
    # EDGES + TRAZABILIDAD
    # ────────────────────────────────────────────────────────────
    edge_list = []
    edge_features = []
    edge_to_flows = {}
    edge_to_labels = {}

    # Agrupar flows por par (src, dst)
    grouped = window_df.groupby([src_col, dst_col])

    for (src_ip, dst_ip), flows_df in grouped:

        src_idx = local_ip_to_idx[src_ip]
        dst_idx = local_ip_to_idx[dst_ip]

        edge_list.append([src_idx, dst_idx])
        edge_id = len(edge_list) - 1

        # Features agregadas
        agg_features = compute_edge_features(flows_df)
        edge_features.append(agg_features)

        # Trazabilidad
        flow_ids = flows_df.index.tolist() if hasattr(flows_df, 'index') else list(range(len(flows_df)))
        edge_to_flows[edge_id] = flow_ids

        flow_labels = flows_df[label_col].tolist()
        edge_to_labels[edge_id] = flow_labels

    # ────────────────────────────────────────────────────────────
    # LABEL de ventana
    # ────────────────────────────────────────────────────────────
    has_attack = (window_df[label_col] != 0).any()
    y = torch.tensor([1 if has_attack else 0], dtype=torch.long)

    if has_attack:
        attack_flows = (window_df[label_col] != 0).sum()
        attack_ratio = attack_flows / len(window_df)
        attack_types = window_df[window_df[label_col] != 0][label_col].unique().tolist()
    else:
        attack_flows = 0
        attack_ratio = 0.0
        attack_types = []

    # ────────────────────────────────────────────────────────────
    # Crear Data object
    # ────────────────────────────────────────────────────────────
    if len(edge_list) == 0:
        return None

    data = Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_list, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_features, dtype=torch.float),
        y=y,
        num_nodes=len(active_ips)
    )

    # Metadata
    data.window_id = window_id
    data.window_start = window_start
    data.window_end = window_end
    data.num_flows = len(window_df)
    data.attack_flows = attack_flows
    data.attack_ratio = attack_ratio
    data.attack_types = attack_types
    data.local_to_global = local_to_global
    data.edge_to_flows = edge_to_flows
    data.edge_to_labels = edge_to_labels

    return data

# Pipeline

In [12]:
# ============================================================================
# PIPELINE PRINCIPAL: Generación de grafos (CON VERBOSE DETALLADO)
# ============================================================================

def generate_graphs_dynamic():
    """
    Pipeline completo de generación de grafos dinámicos
    CON verbose detallado para monitoreo
    """

    print("\n" + "="*70)
    print("🚀 GENERACIÓN DE GRAFOS DINÁMICOS")
    print("="*70)

    # ────────────────────────────────────────────────────────────
    # FASE 1: Crear mapeo global de IPs
    # ────────────────────────────────────────────────────────────
    print("\n1️⃣  Creando mapeo global de IPs...")
    print("   (Leyendo todos los chunks para identificar IPs únicas)")

    global_ip_to_id = {}
    next_id = 0
    chunk_count = 0

    start_time = time.time()

    for chunk in read_filtered_chunks(FILTERED_CHUNKS_FILE):

        chunk_count += 1
        ips_before = len(global_ip_to_id)

        # Procesar IPs
        for ip in chunk[SRC_IP_COL].unique():
            if ip not in global_ip_to_id:
                global_ip_to_id[ip] = next_id
                next_id += 1

        for ip in chunk[DST_IP_COL].unique():
            if ip not in global_ip_to_id:
                global_ip_to_id[ip] = next_id
                next_id += 1

        ips_after = len(global_ip_to_id)
        new_ips = ips_after - ips_before

        # Verbose cada chunk
        print(f"   Chunk {chunk_count:>3}: {len(chunk):>7,} flows | "
              f"IPs únicas: {len(global_ip_to_id):>6,} (+{new_ips:>4})",
              end='\r')

        del chunk

    elapsed_phase1 = time.time() - start_time

    print(f"\n   ✅ Fase 1 completada en {elapsed_phase1:.1f}s")
    print(f"   ✅ Total IPs únicas: {len(global_ip_to_id):,}")
    print(f"   ✅ Total chunks procesados: {chunk_count}")

    # Guardar mapeo
    with open(f'{OUTPUT_DIR}/global_ip_to_id.pkl', 'wb') as f:
        pickle.dump(global_ip_to_id, f)
    print(f"   💾 Guardado: global_ip_to_id.pkl")

    # ────────────────────────────────────────────────────────────
    # FASE 2: Determinar rango temporal
    # ────────────────────────────────────────────────────────────
    print("\n2️⃣  Determinando rango temporal...")

    start_time = time.time()

    dataset_start = None
    dataset_end = None
    chunk_count = 0

    for chunk in read_filtered_chunks(FILTERED_CHUNKS_FILE):

        chunk_count += 1

        # Convertir timestamps
        ns_vals = chunk[TIMESTAMP_COL].values.astype('int64')
        chunk[TIMESTAMP_COL] = pd.to_datetime(ns_vals * 1e6, unit='ns')

        chunk_start = chunk[TIMESTAMP_COL].min()
        chunk_end = chunk[TIMESTAMP_COL].max()

        if dataset_start is None or chunk_start < dataset_start:
            dataset_start = chunk_start
        if dataset_end is None or chunk_end > dataset_end:
            dataset_end = chunk_end

        print(f"   Chunk {chunk_count:>3}: {chunk_start} → {chunk_end}", end='\r')

        del chunk

    duration = dataset_end - dataset_start
    elapsed_phase2 = time.time() - start_time

    print(f"\n   ✅ Fase 2 completada en {elapsed_phase2:.1f}s")
    print(f"   📅 Start: {dataset_start}")
    print(f"   📅 End:   {dataset_end}")
    print(f"   ⏱️  Duration: {duration}")

    # ────────────────────────────────────────────────────────────
    # FASE 3: Acumular flows en ventanas
    # ────────────────────────────────────────────────────────────
    print(f"\n3️⃣  Acumulando flows en ventanas de {WINDOW_SIZE}...")
    print(f"   Window step: {WINDOW_STEP}")

    start_time = time.time()

    window_buffer = defaultdict(list)
    chunk_count = 0
    total_flows = 0

    for chunk in read_filtered_chunks(FILTERED_CHUNKS_FILE):

        chunk_count += 1

        # Convertir timestamps
        ns_vals = chunk[TIMESTAMP_COL].values.astype('int64')
        chunk[TIMESTAMP_COL] = pd.to_datetime(ns_vals * 1e6, unit='ns')

        # Asignar a ventanas
        chunk['window_id'] = ((chunk[TIMESTAMP_COL] - dataset_start) // WINDOW_STEP).astype(int)

        # Agregar a buffer
        for wid in chunk['window_id'].unique():
            window_chunk = chunk[chunk['window_id'] == wid].copy()
            window_buffer[wid].append(window_chunk)

        total_flows += len(chunk)

        # Verbose detallado
        print(f"   Chunk {chunk_count:>3}: "
              f"{len(chunk):>7,} flows | "
              f"Total: {total_flows:>10,} | "
              f"Ventanas: {len(window_buffer):>4}",
              end='\r')

        del chunk

    elapsed_phase3 = time.time() - start_time

    print(f"\n   ✅ Fase 3 completada en {elapsed_phase3/60:.1f} min")
    print(f"   ✅ Total flows procesados: {total_flows:,}")
    print(f"   ✅ Ventanas detectadas: {len(window_buffer)}")

    # Estadísticas de ventanas
    window_sizes = [sum(len(chunk) for chunk in chunks) for chunks in window_buffer.values()]
    print(f"\n   📊 Distribución de flows por ventana:")
    print(f"      Min:    {min(window_sizes):,}")
    print(f"      Max:    {max(window_sizes):,}")
    print(f"      Media:  {sum(window_sizes)/len(window_sizes):,.0f}")
    print(f"      Mediana: {sorted(window_sizes)[len(window_sizes)//2]:,}")

    # ────────────────────────────────────────────────────────────
    # FASE 4: Crear grafos
    # ────────────────────────────────────────────────────────────
    print(f"\n4️⃣  Creando grafos...")
    print(f"   (Esto tomará más tiempo - procesando {len(window_buffer)} ventanas)")

    start_time = time.time()

    graphs_metadata = []
    skipped = 0
    last_print_time = time.time()

    output_file = f'{OUTPUT_DIR}/graphs_dynamic.pkl'

    with open(output_file, 'wb') as f_out:

        for i, window_id in enumerate(sorted(window_buffer.keys())):

            # Concatenar chunks de esta ventana
            window_df = pd.concat(window_buffer[window_id], ignore_index=True)

            # Límites temporales
            window_start = dataset_start + window_id * WINDOW_STEP
            window_end = window_start + WINDOW_SIZE

            # Filtrar por ventana exacta
            window_df = window_df[
                (window_df[TIMESTAMP_COL] >= window_start) &
                (window_df[TIMESTAMP_COL] < window_end)
            ]

            # Descartar ventanas muy pequeñas
            if len(window_df) < 5:
                skipped += 1
                continue

            # Crear grafo
            graph = create_graph_dynamic(
                window_df, window_id, window_start, window_end,
                SRC_IP_COL, DST_IP_COL, LABEL_COL, global_ip_to_id
            )

            if graph is not None:
                pickle.dump(graph, f_out)
                graphs_metadata.append({
                    'window_id': window_id,
                    'label': graph.y.item(),
                    'num_nodes': graph.num_nodes,
                    'num_edges': graph.edge_index.shape[1],
                    'num_flows': graph.num_flows,
                    'attack_ratio': graph.attack_ratio
                })

            # ────────────────────────────────────────────────────
            # Verbose cada 10 grafos O cada 30 segundos
            # ────────────────────────────────────────────────────
            current_time = time.time()

            if (i + 1) % 10 == 0 or (current_time - last_print_time) > 30:

                elapsed = current_time - start_time
                progress = (i + 1) / len(window_buffer) * 100
                est_total = elapsed / (i + 1) * len(window_buffer)
                est_remaining = est_total - elapsed

                # Estadísticas de los últimos 10 grafos
                recent = graphs_metadata[-10:] if len(graphs_metadata) >= 10 else graphs_metadata
                avg_nodes = sum(g['num_nodes'] for g in recent) / len(recent) if recent else 0
                avg_edges = sum(g['num_edges'] for g in recent) / len(recent) if recent else 0
                attacks_recent = sum(1 for g in recent if g['label'] == 1)

                print(f"\n   Progreso: {i+1}/{len(window_buffer)} ({progress:.1f}%)")
                print(f"      Grafos creados: {len(graphs_metadata)}")
                print(f"      Omitidos: {skipped}")
                print(f"      Ataques: {sum(1 for g in graphs_metadata if g['label']==1)} "
                      f"({attacks_recent}/10 últimos)")
                print(f"      Avg nodos: {avg_nodes:.0f} | Avg aristas: {avg_edges:.0f}")
                print(f"      Tiempo: {elapsed/60:.1f} min | Restante: ~{est_remaining/60:.1f} min")

                last_print_time = current_time

            # Verbose compacto cada grafo
            label_symbol = "🔴" if graph and graph.y.item() == 1 else "🟢"
            print(f"   Ventana {window_id:>4} {label_symbol}: "
                  f"{len(graphs_metadata):>4} grafos | "
                  f"{skipped:>3} omitidos",
                  end='\r')

            del window_df

    elapsed_phase4 = time.time() - start_time

    print(f"\n\n   ✅ Fase 4 completada en {elapsed_phase4/60:.1f} min")
    print(f"   ✅ Grafos creados: {len(graphs_metadata)}")
    print(f"   ✅ Ventanas omitidas: {skipped}")

    # ────────────────────────────────────────────────────────────
    # FASE 5: Estadísticas finales y guardar metadata
    # ────────────────────────────────────────────────────────────
    print(f"\n5️⃣  Generando estadísticas finales...")

    stats_df = pd.DataFrame(graphs_metadata)

    print(f"\n{'='*70}")
    print("✅ GENERACIÓN COMPLETADA")
    print(f"{'='*70}")

    print(f"\n📊 RESUMEN GENERAL:")
    print(f"   {'─'*66}")
    print(f"   Total ventanas detectadas:  {len(window_buffer):>6,}")
    print(f"   Grafos creados:             {len(graphs_metadata):>6,}")
    print(f"   Ventanas omitidas:          {skipped:>6,} (< 5 flows)")
    print(f"   {'─'*66}")

    print(f"\n📊 DISTRIBUCIÓN DE LABELS:")
    print(f"   {'─'*66}")
    normal_count = (stats_df['label']==0).sum()
    attack_count = (stats_df['label']==1).sum()
    print(f"   Normal (0):  {normal_count:>6,} ({normal_count/len(stats_df)*100:>5.1f}%)")
    print(f"   Ataque (1):  {attack_count:>6,} ({attack_count/len(stats_df)*100:>5.1f}%)")
    print(f"   {'─'*66}")

    print(f"\n📊 ESTADÍSTICAS DE NODOS:")
    print(f"   {'─'*66}")
    print(f"   Mínimo:  {stats_df['num_nodes'].min():>6,}")
    print(f"   Máximo:  {stats_df['num_nodes'].max():>6,}")
    print(f"   Media:   {stats_df['num_nodes'].mean():>6,.0f}")
    print(f"   Mediana: {stats_df['num_nodes'].median():>6,.0f}")
    print(f"   {'─'*66}")

    print(f"\n📊 ESTADÍSTICAS DE ARISTAS:")
    print(f"   {'─'*66}")
    print(f"   Mínimo:  {stats_df['num_edges'].min():>6,}")
    print(f"   Máximo:  {stats_df['num_edges'].max():>6,}")
    print(f"   Media:   {stats_df['num_edges'].mean():>6,.0f}")
    print(f"   Mediana: {stats_df['num_edges'].median():>6,.0f}")
    print(f"   {'─'*66}")

    print(f"\n📊 ESTADÍSTICAS DE FLOWS:")
    print(f"   {'─'*66}")
    print(f"   Mínimo:  {stats_df['num_flows'].min():>6,}")
    print(f"   Máximo:  {stats_df['num_flows'].max():>6,}")
    print(f"   Media:   {stats_df['num_flows'].mean():>6,.0f}")
    print(f"   Mediana: {stats_df['num_flows'].median():>6,.0f}")
    print(f"   {'─'*66}")

    print(f"\n📊 GRAFOS CON ATAQUES:")
    print(f"   {'─'*66}")
    attack_graphs = stats_df[stats_df['label'] == 1]
    if len(attack_graphs) > 0:
        print(f"   Flows de ataque (promedio): {attack_graphs['attack_flows'].mean():>6,.0f}")
        print(f"   Ratio de ataque (promedio): {attack_graphs['attack_ratio'].mean()*100:>6.1f}%")
        print(f"   Ratio mínimo:               {attack_graphs['attack_ratio'].min()*100:>6.1f}%")
        print(f"   Ratio máximo:               {attack_graphs['attack_ratio'].max()*100:>6.1f}%")
    print(f"   {'─'*66}")

    # Guardar metadata
    metadata = {
        'dataset_start': dataset_start,
        'dataset_end': dataset_end,
        'duration': duration,
        'window_size': WINDOW_SIZE,
        'window_step': WINDOW_STEP,
        'total_ips': len(global_ip_to_id),
        'total_graphs': len(graphs_metadata),
        'graphs_normal': normal_count,
        'graphs_attack': attack_count,
        'node_features_dim': 18,
        'edge_features_dim': 37,
        'statistics': stats_df.to_dict('records')
    }

    with open(f'{OUTPUT_DIR}/metadata.pkl', 'wb') as f:
        pickle.dump(metadata, f)

    print(f"\n📁 ARCHIVOS GENERADOS:")
    print(f"   {'─'*66}")

    # Calcular tamaños de archivo
    graphs_size = os.path.getsize(output_file) / (1024**2)  # MB
    mapping_size = os.path.getsize(f'{OUTPUT_DIR}/global_ip_to_id.pkl') / 1024  # KB
    metadata_size = os.path.getsize(f'{OUTPUT_DIR}/metadata.pkl') / 1024  # KB

    print(f"   📄 graphs_dynamic.pkl")
    print(f"      {len(graphs_metadata)} grafos | {graphs_size:.1f} MB")
    print(f"   📄 global_ip_to_id.pkl")
    print(f"      {len(global_ip_to_id):,} IPs | {mapping_size:.1f} KB")
    print(f"   📄 metadata.pkl")
    print(f"      Estadísticas completas | {metadata_size:.1f} KB")
    print(f"   {'─'*66}")

    print(f"\n⏱️  TIEMPOS DE EJECUCIÓN:")
    print(f"   {'─'*66}")
    total_time = elapsed_phase1 + elapsed_phase2 + elapsed_phase3 + elapsed_phase4
    print(f"   Fase 1 (Mapeo IPs):        {elapsed_phase1:>6.1f}s ({elapsed_phase1/total_time*100:>5.1f}%)")
    print(f"   Fase 2 (Rango temporal):   {elapsed_phase2:>6.1f}s ({elapsed_phase2/total_time*100:>5.1f}%)")
    print(f"   Fase 3 (Acumular ventanas): {elapsed_phase3/60:>6.1f}m ({elapsed_phase3/total_time*100:>5.1f}%)")
    print(f"   Fase 4 (Crear grafos):     {elapsed_phase4/60:>6.1f}m ({elapsed_phase4/total_time*100:>5.1f}%)")
    print(f"   {'─'*66}")
    print(f"   TOTAL:                     {total_time/60:>6.1f} min")
    print(f"   {'─'*66}")

    print(f"\n✅ Ubicación: {OUTPUT_DIR}")
    print(f"{'='*70}\n")

    return graphs_metadata


In [13]:
# ============================================================================
# EJECUTAR GENERACIÓN
# ============================================================================

if __name__ == '__main__':

    # Verificar que archivos necesarios existen
    if not os.path.exists(FILTERED_CHUNKS_FILE):
        print(f"❌ ERROR: No se encuentra {FILTERED_CHUNKS_FILE}")
        print("   Primero debes crear filtered_chunks.pkl")
    else:
        # Ejecutar
        start_time = time.time()

        graphs_metadata = generate_graphs_dynamic()

        elapsed = time.time() - start_time
        print(f"\n⏱️  Tiempo total: {elapsed/60:.1f} minutos")

        print("\n✅ PROCESO COMPLETADO")


🚀 GENERACIÓN DE GRAFOS DINÁMICOS

1️⃣  Creando mapeo global de IPs...
   (Leyendo todos los chunks para identificar IPs únicas)
   Chunk  81: 115,529 flows | IPs únicas: 205,801 (+ 644)
   ✅ Fase 1 completada en 108.6s
   ✅ Total IPs únicas: 205,801
   ✅ Total chunks procesados: 81
   💾 Guardado: global_ip_to_id.pkl

2️⃣  Determinando rango temporal...
   Chunk  81: 2018-02-22 12:25:10.148000 → 2018-03-02 20:04:25.524000
   ✅ Fase 2 completada en 72.4s
   📅 Start: 2018-02-14 12:28:07.704999936
   📅 End:   2018-03-02 20:04:25.524000
   ⏱️  Duration: 16 days 07:36:17.819000064

3️⃣  Acumulando flows en ventanas de 0 days 00:10:00...
   Window step: 0 days 00:10:00
   Chunk  81: 115,529 flows | Total: 20,115,529 | Ventanas:  736
   ✅ Fase 3 completada en 1.7 min
   ✅ Total flows procesados: 20,115,529
   ✅ Ventanas detectadas: 736

   📊 Distribución de flows por ventana:
      Min:    1
      Max:    249,267
      Media:  27,331
      Mediana: 30,270

4️⃣  Creando grafos...
   (Esto toma

KeyboardInterrupt: 